COMP 215 - LAB 3 Classes (NEO)
----------------
#### Name:
#### Date:

This lab exercise introduces `class` as a means of organizing related data and functions.

**Building on new concepts from lab 2**:
  * a `record` is a related collection of data, with fields for each data value
  * an `API` is an "Application Programmers Interface" defining how a programmer interacts with a system.
  * *f-string* simplifies string formatting operations

**New Python Concepts**:
  * the `class` keyword allows you define a new data `type`, with a set of operations on that data.
  * a `dataclass` simplifies class definition for classes that primarily encapsulate a data structure.

As usual, the first code cell simply imports all the modules we'll be using...

In [13]:
import datetime, json, requests
from pprint import pprint    # Pretty Print - built-in python function to nicely format data structures

We'll continue working with [Near Earth Object](https://cneos.jpl.nasa.gov/) data
> using NASA's API:  [https://api.nasa.gov/](https://api.nasa.gov/#NeoWS)

Here's a brief review from Lab 2 on how to use it...

### Review: making a query

Here's a query that gets the record for a single NEO that recently passed by.

In [14]:
API_KEY = 'ToSKtbouoLPpBUdmQHKIqSJzPffVwZZWmyrOHaYQ'  # substitute your API key here

def get_neos(start_date):
    """ Return a list of NEO for the week starting at start_date """
    url = f'https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&api_key={API_KEY}'
    # Fetch last week's NEO feed
    response = requests.request("GET", url, headers={}, data={})
    data = json.loads(response.text)
    return [neo for dated_records in data['near_earth_objects'].values() for neo in dated_records ]

def get_neo(id):
    """ Return a NEO record for the given id """
    url = f'https://api.nasa.gov/neo/rest/v1/neo/{id}?api_key={API_KEY}'
    response = requests.request("GET", url, headers={}, data={})
    return json.loads(response.text)

# Sample usage:  get the list of NEOs for a given week, then lookup the latest NEO record in that list.
week_start = '2023-01-15'
neos = get_neos(week_start)
print(f'{len(neos)} Near Earth Objects found for week of {week_start}')
assert len(neos) > 0, f'Oh oh!  No NEOs found for {week_start}'
neo = get_neo(neos[-1]['id'])  # get the very latest NEO
pprint(neo)

13 Near Earth Objects found for week of 2023-01-15
{'absolute_magnitude_h': 27.9,
 'close_approach_data': [{'close_approach_date': '1963-10-03',
                          'close_approach_date_full': '1963-Oct-03 10:42',
                          'epoch_date_close_approach': -197126280000,
                          'miss_distance': {'astronomical': '0.0314084333',
                                            'kilometers': '4698634.721717071',
                                            'lunar': '12.2178805537',
                                            'miles': '2919596.2325987398'},
                          'orbiting_body': 'Venus',
                          'relative_velocity': {'kilometers_per_hour': '52882.6650360426',
                                                'kilometers_per_second': '14.6896291767',
                                                'miles_per_hour': '32859.2315053121'}},
                         {'close_approach_date': '1966-10-20',
                         

## Exercise 1: Define a CloseApproach class

Each NEO record comes with a list of `close_approach_data`, where each record in this list represents a single “close approach” to another orbiting body.
* Develop a class named `CloseApproach` to represent a *single* close approach record.
* State variables are
    * orbiting body (`str`)
    * approach date (`datetime` object!)
    * miss distance (`float` in km, document it!)
    * relative velocity (`float` in km/hr, ditto)

* Operations must include:
    * `__init__(self, ...)` method to initialize a new object with specific data values
    * `__str__(self)` method to return a nicely formatted string representation of the object.

Write a little code to test your new class.

In [15]:
# Ex. 1 your code here
class CloseApproach():
  def __init__(self, close_approach_data):
    self.orbiting_body = close_approach_data['orbiting_body']
    self.approach_date = close_approach_data['close_approach_date']
    self.miss_distance = close_approach_data['miss_distance']['kilometers']
    self.relative_velocity = close_approach_data['relative_velocity']['kilometers_per_hour']

  def __str__(self):
    return f'Passed by {self.orbiting_body} on {self.approach_date} by {self.miss_distance}km at a speed of {self.relative_velocity}km/hr'

In [16]:
test_data = {'close_approach_date': '1949-07-04',
                       'close_approach_date_full': '1949-Jul-04 22:19',
                       'epoch_date_close_approach': -646710060000,
                       'miss_distance': {'astronomical': '0.419726307',
                                         'kilometers': '62790161.51016609',
                                         'lunar': '163.273533423',
                                         'miles': '39015997.166588442'},
                       'orbiting_body': 'Earth',
                      'relative_velocity': {'kilometers_per_hour': '73509.6739172087',
                                            'kilometers_per_second': '20.4193538659',
                                            'miles_per_hour': '45676.0526626122'}}

test_approach = CloseApproach(test_data)

print(test_approach)

Passed by Earth on 1949-07-04 by 62790161.51016609km at a speed of 73509.6739172087km/hr


## Exercise 2: Factory function: get_close_approach

We want to be able to construct CloseApproach objects easily from a data record returned from the NEO API.  

Write an "object factory" function...   

    def get_close_approach(record):
        ...

This function provides an easy way to create a `CloseApproach` instance.  It takes a dictionary for a single `close_approach_data` record, constructs and returns a `CloseApproach` object representing that same record.
This kind of function is called a “Factory” because it handles the details of constructing an object from raw materials.

Remember to convert each element from the data dictionary to the correct type (e.g., parse the date/time string into a `datetime` object).
Add little code to test your new factory function.

In [17]:
# Ex. 2 your code here
def get_close_approach(record):
  '''Takes all the close approach data for a NEO and returns it in a class'''
  return CloseApproach(record)


In [18]:
test_data = {'close_approach_date': '1949-07-04',
                       'close_approach_date_full': '1949-Jul-04 22:19',
                       'epoch_date_close_approach': -646710060000,
                       'miss_distance': {'astronomical': '0.419726307',
                                         'kilometers': '62790161.51016609',
                                         'lunar': '163.273533423',
                                         'miles': '39015997.166588442'},
                       'orbiting_body': 'Earth',
                      'relative_velocity': {'kilometers_per_hour': '73509.6739172087',
                                            'kilometers_per_second': '20.4193538659',
                                            'miles_per_hour': '45676.0526626122'}}

test_approach = get_close_approach(test_data)

print(test_approach)

Passed by Earth on 1949-07-04 by 62790161.51016609km at a speed of 73509.6739172087km/hr


## Exercise 3:  Define an Asteroid class

Define a simple Asteroid class with some basic state variables representing a single NEO.  Your Asteroid class should define at least 4 "state variables:”
* id  (`int`)
* name (`str`)
* estimated_diameter (`float` in m)
* is_potentially_hazardous (`bool`)
* close_approaches (`list` of CloseApproach objects, default to empty list)

Operations must include:
* `__init__(self, ...)` method to initialize a new object with specific data values
* `__str__(self)` method to return a nicely formatted string representation of the object.

Write a little code to test you new class (just leave close_approaches as an empty list for now).

In [19]:
# Ex. 3 your code here
class Astroid():
  def __init__(self, neo_data):
    self.id = neo_data['id']
    self.name = neo_data['name']
    self.estimated_diameter = neo_data['estimated_diameter']['meters']
    self.is_potentially_hazardous = neo_data['is_potentially_hazardous_asteroid']
    self.close_approaches = self.set_close_approach_list(neo_data['close_approach_data'])

  def __str__(self):
    return f'NEO id: {self.id} \nNEO name: {self.name} \nNEO Diameter: {self.estimated_diameter['estimated_diameter_min']}m - {self.estimated_diameter['estimated_diameter_max']}m \nIs NEO hazardous? {self.is_potentially_hazardous} \nNEO approach data {self.close_approaches}'

  def set_close_approach_list(self, close_approaches):
    '''Runs when initializing Astroids close aproaches and returns a list of approaches to be put into the approach variable'''
    approaches = []
    for approach in close_approaches:
      approaches.append(CloseApproach(approach))

    return approaches

  def nearest_miss(self):
    '''Returns the nearest miss to Earth by the astroid'''
    closest_approach = float('inf')
    for approach in self.close_approaches:
      if closest_approach == float('inf') and approach.orbiting_body == 'Earth':
        closest_approach = approach

      elif float(closest_approach.miss_distance) > float(approach.miss_distance) and approach.orbiting_body == 'Earth':
        closest_approach = approach

    return closest_approach

  def estimated_mass(self):
    '''Using Newtons law of gravitation figure out mass of astroid in relation to earth'''
    #Get the know variables for the equation (all but the orbit distance)
    earth_mass = 5.972 * 10**24
    g_constant = 6.6743 * 10**-11 #will have to measure distance in meters and weights in kilograms



In [20]:
test_astroid = Astroid(neo)
print(test_astroid)

NEO id: 3582523 
NEO name: (2011 UC64) 
NEO Diameter: 6.9912523225m - 15.6329154409m 
Is NEO hazardous? False 
NEO approach data [<__main__.CloseApproach object at 0x7d9e901fe0f0>, <__main__.CloseApproach object at 0x7d9e901fe2a0>, <__main__.CloseApproach object at 0x7d9e901feae0>, <__main__.CloseApproach object at 0x7d9e901fe9f0>, <__main__.CloseApproach object at 0x7d9e901fec30>, <__main__.CloseApproach object at 0x7d9e901fe9c0>, <__main__.CloseApproach object at 0x7d9e901fea80>, <__main__.CloseApproach object at 0x7d9e901fddc0>, <__main__.CloseApproach object at 0x7d9e901fea20>, <__main__.CloseApproach object at 0x7d9e901feb10>, <__main__.CloseApproach object at 0x7d9e901feb70>, <__main__.CloseApproach object at 0x7d9e901fc170>, <__main__.CloseApproach object at 0x7d9e901fc050>, <__main__.CloseApproach object at 0x7d9e901fc3e0>, <__main__.CloseApproach object at 0x7d9e901fc080>, <__main__.CloseApproach object at 0x7d9e901fc230>, <__main__.CloseApproach object at 0x7d9e901fcbf0>, <__

## Exercise 4: Asteroid factory

Write a function that returns an Asteroid object just from the id for a single NEO.

    def asteroid_from_neo(neo_id):
        ...

This factory function takes the `id` for a single NEO, fetches the NEO record from API, constructs and returns an Asteroid object representing that NEO.  *Hint*: I provided the code fetch a NEO from its `id` above.

Every `Asteroid` should have a list of “close approaches”.
*Hint*: use the `get_close_approach` factory you defined above to construct the required list of CloseApproach objects.

Now add a new method to `Asteroid` class to return the `CloseApproach` object from the asteroid representing its nearest to **Earth**:

    def nearest_miss(self):
        ...

Extend your test code to demonstrate these new features.

In [21]:
def asteroid_from_neo(neo_id):
  return Astroid(get_neo(neo_id))

test_astroid = asteroid_from_neo('3653388')
assert test_astroid.id == '3653388'

In [22]:
# Ex. 4 your test code for nearest_miss here
test_astroid = asteroid_from_neo('3653388')
test_astroid.nearest_miss()

## Challenge - Take your skills to the next level...
### Exercise 5: develop a useful analysis / data product

 With these data structures in place, we can now start answering all kinds of interesting questions about a single Asteroid or a set of Asteroids.  
Here’s a couple ideas to try:

* write a **function** named `most_dangerous_approach`, that takes a date range and returns a single “potentially hazardous” Asteroid object that makes the closest approach to Earth within that range.  Your algorithm will ultimately need to:
    * grab the list of NEO’s for the given date range;
    * use a list comprehension to build the list of Asteroid objects for the NEO’s returned
    * use a list comprehension to filter  potentially hazardous Asteroids only;
    * use a list comprehension to map each Asteroid to its  nearest_miss
    * apply Python’s min function to identify the Asteroid with the nearest_miss

You may want to decompose some of these steps into smaller functions.
* add a method to the Asteroid class, `estimated_mass`, that computes an estimate of the Asteroid’s mass based on its diameter.  This is a model – state your assumptions.
* add a method to the `CloseApproach` class, `impact_force`,  that estimates the force of impact if the Asteroid hit the orbiting object.  Again, this is a model, state your assumptions.

In [23]:
# Ex. 5 (challenge) your code here
def most_dangerous_approach():


SyntaxError: incomplete input (ipython-input-180677562.py, line 3)